# Tutorial: Setting Up the Analysis Environment

This notebook guides you through installing all software required to run the
functional connectivity pipeline on your local machine or HPC cluster.
No prior experience with command-line tools or container technologies is assumed.

---

## 1. Operating Environment

The tutorial has been developed and tested on **Windows 11 with WSL2**,
and is fully compatible with native **Linux** and **macOS** environments.

| Platform | Recommended setup |
|---|---|
| Windows 10 / 11 | Enable WSL2 (Ubuntu 22.04 recommended) |
| macOS 12+ | Use the built-in Terminal app |
| Linux (Ubuntu 20.04+) | Use the system terminal |

### Windows: Enabling WSL2

Open **PowerShell as administrator** and run:

```powershell
wsl --install
```

This installs WSL2 and Ubuntu in a single step. After restarting your machine,
launch **Ubuntu** from the Start menu. All subsequent command-line instructions
in this tutorial should be executed inside the WSL2 terminal, not in
Command Prompt or PowerShell.

> **Note for institutional computers:** If you work in a hospital or clinical
> setting with IT restrictions, WSL2 and Docker may require administrator
> approval. Contact your IT department before proceeding.

---

## 2. Required Software

Install the following tools in the order listed. Each section explains what
the tool does and provides the installation command for each platform.

---

### 2.1 Git

Git is a version control system used to track changes to code and to download
(clone) the project repository from GitHub.

**Ubuntu / WSL2:**
```bash
sudo apt update && sudo apt install git
git --version   # expected: git version 2.x.x
```

**macOS:**
```bash
# Git is included with Xcode Command Line Tools
xcode-select --install
```

**Windows (native, optional):**  
Download the installer from https://git-scm.com. If you are using WSL2,
the Ubuntu installation above is sufficient.

---

### 2.2 Anaconda / Miniconda

Anaconda provides the `conda` package manager, which is used to create an
isolated Python environment for running Snakemake on the host machine.
Miniconda is a lightweight alternative that installs only `conda` itself;
it is recommended if disk space is limited.

**All platforms:**  
Download the Miniconda installer from https://docs.conda.io/en/latest/miniconda.html
and follow the platform-specific instructions. After installation, verify the setup:

```bash
conda --version   # expected: conda 23.x.x or later
```

> **Anaconda vs Miniconda:** If you already have Anaconda installed, there is no
> need to install Miniconda. Both provide the same `conda` command.

---

### 2.3 Docker Desktop

Docker is the container platform used to build and run the reproducible
analysis environment. Think of a Docker container as a self-contained
virtual laboratory: it packages the operating system, Python interpreter,
and every library used in the analysis into a single portable unit.
Any researcher who runs the same container image will obtain identical results,
regardless of what is installed on their own machine.

**Windows / macOS:**  
Download Docker Desktop from https://www.docker.com and run the installer.
On Windows, Docker Desktop integrates with WSL2 and will prompt for any
additional configuration on first launch.

**Ubuntu / WSL2:**
```bash
# Add Docker's official repository
sudo apt update
sudo apt install ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg \
     -o /etc/apt/keyrings/docker.asc

echo "deb [arch=$(dpkg --print-architecture) \
  signed-by=/etc/apt/keyrings/docker.asc] \
  https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo $VERSION_CODENAME) stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

sudo apt update
sudo apt install docker-ce docker-ce-cli containerd.io

# Allow running Docker without sudo
sudo usermod -aG docker $USER
newgrp docker

# Verify installation
docker --version   # expected: Docker version 24.x.x or later
docker run hello-world
```

---

### 2.4 Snakemake

Snakemake is the workflow manager that orchestrates the analysis pipeline.
It runs on the host machine (outside the container) and calls the Docker
container for each computational step. Installing Snakemake in a dedicated
conda environment avoids conflicts with other Python packages.

```bash
# Create a minimal environment for Snakemake
conda env create -f environment_snakemake.yml

# Activate it
conda activate snakemake-host

# Verify
snakemake --version   # expected: 8.x.x
```

The file `environment_snakemake.yml` is included in the project repository.
You will need to activate this environment every time you run the pipeline.

---

### 2.5 Node.js and bids-validator

bids-validator is a command-line tool that checks whether a dataset conforms
to the BIDS specification. It is written in JavaScript and requires Node.js.

**Ubuntu / WSL2:**
```bash
# Install Node.js via the NodeSource repository (LTS version)
curl -fsSL https://deb.nodesource.com/setup_lts.x | sudo -E bash -
sudo apt install nodejs
node --version   # expected: v20.x.x or later

# Install bids-validator globally
npm install -g bids-validator
bids-validator --version
```

**macOS:**
```bash
# Using Homebrew
brew install node
npm install -g bids-validator
```

**Windows (native):**  
Download the Node.js LTS installer from https://nodejs.org, then open
Command Prompt and run `npm install -g bids-validator`.

---

## 3. Clone the Repository

Once all software is installed, clone the project repository:

```bash
git clone https://github.com/HanweiLiu-Viola/Causality-Analysis.git
cd Causality-Analysis
```

The repository contains all source code, the Snakefile, and the notebook
tutorials. Large data files and container images are not stored in the
repository; they are generated or pulled automatically when you run the
pipeline in the following sections.

---

## 4. Verify the Setup

Run the following checklist to confirm that all tools are correctly installed
before proceeding to the next tutorial notebook.


In [ ]:
import subprocess
import shutil

checks = {
    "git":            ["git", "--version"],
    "conda":          ["conda", "--version"],
    "docker":         ["docker", "--version"],
    "snakemake":      ["snakemake", "--version"],
    "node":           ["node", "--version"],
    "bids-validator": ["bids-validator", "--version"],
}

print(f"{'Tool':<20} {'Status':<10} {'Version'}")
print("-" * 55)
for name, cmd in checks.items():
    if shutil.which(cmd[0]) is None:
        print(f"{name:<20} {'NOT FOUND':<10}")
    else:
        result = subprocess.run(cmd, capture_output=True, text=True)
        version = (result.stdout or result.stderr).strip().split("\n")[0]
        print(f"{name:<20} {'OK':<10} {version}")

All six tools should show **OK**. If any show **NOT FOUND**, return to the
corresponding installation section above before continuing.

---

**Next:** [01_data_standardization.ipynb](01_data_standardization.ipynb)